In [ ]:
from pathlib import Path
import json
import platform

import matplotlib.pyplot as plt
import pandas as pd
import sklearn
from IPython.display import Image, display

from research_pipeline import run_all

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

In [ ]:
results = run_all(
    csv_path=Path("Student_Depression_Dataset.csv"),
    artifact_root=Path("artifacts"),
)
print("Selected models:", results["selected"])

In [ ]:
display(results["audit"]["audit"])
display(results["audit"]["health"].style.format({"depression_rate": "{:.3f}"}))
display(Image(filename="artifacts/figures/figure_1_data_and_proxy_audit.png", width=900))

In [ ]:
performance_columns = [
    "feature_set", "model", "AUROC", "AUROC_low", "AUROC_high",
    "AUPRC", "AUPRC_low", "AUPRC_high", "Brier",
    "Balanced_accuracy", "Sensitivity", "Specificity", "F1"
]
display(
    results["summary"][performance_columns]
    .sort_values(["feature_set", "AUPRC"], ascending=[True, False])
    .style.format(precision=3)
)
display(Image(filename="artifacts/figures/figure_2_nested_cv_auprc.png", width=900))
display(Image(filename="artifacts/figures/figure_3_discrimination_curves.png", width=900))
display(Image(filename="artifacts/figures/figure_4_calibration.png", width=550))

In [ ]:
display(results["geo_overall"].style.format(precision=3))
display(
    results["geo_detail"][[
        "feature_set", "model", "held_out_division", "n", "prevalence",
        "AUROC", "AUPRC", "Brier", "Sensitivity", "Specificity"
    ]].style.format(precision=3)
)
display(Image(filename="artifacts/figures/figure_5_geographic_transportability.png", width=900))

In [ ]:
gaps = pd.read_csv("artifacts/tables/subgroup_metric_gaps.csv")
display(gaps.style.format(precision=3))

gender_rows = results["fairness"][results["fairness"]["group"] == "Gender"]
display(gender_rows.style.format(precision=3))

In [ ]:
display(results["conformal_runs"].style.format(precision=3))
display(results["conformal_runs"].mean(numeric_only=True).to_frame("mean").T.style.format(precision=3))
display(Image(filename="artifacts/figures/figure_6_selective_prediction.png", width=750))

In [ ]:
required_files = [
    "artifacts/results/run_manifest.json",
    "artifacts/results/oof_predictions.csv",
    "artifacts/tables/nested_cv_performance.csv",
    "artifacts/tables/geographic_validation_overall.csv",
    "artifacts/tables/subgroup_fairness_metrics.csv",
    "artifacts/tables/conformal_runs.csv",
]
for filename in required_files:
    path = Path(filename)
    assert path.exists() and path.stat().st_size > 0, filename

assert len(results["raw"]) == 4464
assert int(results["y"].sum()) == 1201
assert results["predictions"].groupby(["feature_set", "model"]).size().eq(4464).all()
assert results["predictions"]["probability"].between(0, 1).all()
assert set(results["selected"]) == {"full", "proxy_free"}

with open("artifacts/results/run_manifest.json", encoding="utf-8") as handle:
    manifest = json.load(handle)
display(manifest["selected_models"])
print("All reproducibility checks passed.")